# Workshop 3 · 03 — CV-Teaser: Wagon-Schaden → Werkstatt (Aufgabe)

**Ziel:** Der unstrukturierteste Datentyp — ein **Bild**. Eine **multimodale** AI Function bekommt den Foto-Pfad und liefert pro Wagon einen strukturierten Schadensbericht (Typ, Schweregrad, Empfehlung, UIC per OCR). Daraus leitest du den **Werkstatt-Auftrag** ab.

**Ausgangslage:** Wagon-Fotos in `Files/wagon_photos_w3` (aus Notebook 00).

**Deine Aufgaben:**
1. Die Fotos als Pfad-Spalte laden (Zelle unten ist bereitgestellt — der OneLake-`abfss://`-Pfad ist der Stolperstein).
2. Mit `ai.generate_response` (JSON-Schema, `col_types={'file_path': 'path'}`) pro Foto einen Schadensbericht erzeugen.
3. JSON parsen, Werkstatt-Zuordnung + `dispatch`-Flag ableiten und als Tabelle `schaden_work_orders` speichern.

Referenz: https://learn.microsoft.com/fabric/data-science/ai-functions/multimodal-overview

## Schritt 1 — Fotos laden (bereitgestellt)
Die AI Function braucht einen voll qualifizierten OneLake-`abfss://`-Pfad (kein relativer Pfad). Führe die Zelle aus.

In [ ]:
import synapse.ml.spark.aifunc as aifunc

# OneLake-ABFSS-Pfad zum Lakehouse bauen
onelake_endpoint = notebookutils.conf.get('trident.onelake.endpoint').replace('https://', '')
workspace_id = notebookutils.runtime.context['currentWorkspaceId']
lakehouse_id = notebookutils.runtime.context['defaultLakehouseId']

base_path = f'abfss://{workspace_id}@{onelake_endpoint}/{lakehouse_id}/Files/wagon_photos_w3'

files_df = aifunc.list_file_paths(base_path)
display(files_df)

## Schritt 2 — Schadensbericht mit multimodaler AI Function

**Aufgabe:**
1. Definiere ein JSON-Schema mit `damage_type` (enum: rost/kratzer/delle/graffiti/riss/kein_schaden), `severity` (integer 1–5), `recommended_action` (string), `uic_number` (string), `safety_relevant` (boolean).
2. Formuliere einen Prompt als Prüfingenieur und rufe `files_df.ai.generate_response(...)` mit `col_types={'file_path': 'path'}`, deinem `response_format` und `output_col='report_json'` auf.

In [ ]:
# This code uses AI. Always review output for mistakes.

# TODO 1: JSON-Schema fuer den Schadensbericht definieren.
report_schema = ...

# TODO 2: Prompt als Pruefingenieur formulieren (Schadenstyp, Schweregrad 1-5, Empfehlung, UIC per OCR).
prompt = ...

# TODO 3: files_df.ai.generate_response(...) mit col_types={'file_path': 'path'} aufrufen und anzeigen.
reports = ...

## Schritt 3 — Werkstatt-Auftrag ableiten und speichern

**Aufgabe:** Parse `report_json`, leite eine `workshop`-Spalte ab (rost/kratzer → Karosserie/Lack, graffiti → Reinigung, delle/riss → Strukturpruefung) und ein `dispatch`-Flag (`severity >= 4` **oder** `safety_relevant`). Speichere als Tabelle `schaden_work_orders`.

*Hinweis:* `from_json(col('report_json'), <StructType>)` zum Parsen, `when(...).otherwise(...)` für die Zuordnung.

In [ ]:
from pyspark.sql.functions import from_json, col, when, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType

# TODO 1: Struktur-Schema passend zu report_json definieren und mit from_json parsen.
# TODO 2: workshop-Zuordnung + dispatch-Flag (severity>=4 ODER safety_relevant) ableiten.
# TODO 3: Ergebnis als Tabelle 'schaden_work_orders' speichern und nach severity sortiert anzeigen.

### Ergebnis-Check & Operationalisierung
Aus Bildern wurde die strukturierte Tabelle `schaden_work_orders` mit Werkstatt-Zuordnung und `dispatch`-Flag. Im echten Workload folgt:
- **Activator**-Regel auf `dispatch = true` → Teams-Nachricht an die Werkstatt-Disposition.
- **Power BI** (Direct Lake) Dashboard über offene Aufträge.